# Coleta de Dados — Vendas de Combustíveis (ANP)

Este notebook coleta os dados de vendas de combustíveis fósseis no Brasil
a partir da Agência Nacional do Petróleo (ANP).

**Fonte:** Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP)  
**Acesso:** Portal de Dados Abertos da ANP  
**Granularidade:** Por tipo de combustível, por UF, por mês  
**Período:** 2000 em diante

In [1]:
import pandas as pd
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas!")

Bibliotecas carregadas!


In [2]:
# ANP disponibiliza vendas de derivados por estado e mês
# URL do arquivo de vendas de combustíveis
url_anp = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos/de/vdpb/vendas-de-derivados-de-petroleo-e-biocombustiveis.xlsx"

r = requests.get(url_anp, timeout=60)
print(f"Status: {r.status_code}")
print(f"Tamanho: {len(r.content):,.0f} bytes")

Status: 404
Tamanho: 26 bytes


In [3]:
# URL correta dos dados de vendas de combustíveis da ANP
url_anp = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos/de/vdpb/vendas-combustiveis-m3.xlsx"

r = requests.get(url_anp, timeout=60)
print(f"Status: {r.status_code}")
print(f"Tamanho: {len(r.content):,.0f} bytes")

# Se não funcionar, tentar URL alternativa
if r.status_code != 200:
    url_anp2 = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/vdpb/vendas-combustiveis-m3.xlsx"
    r = requests.get(url_anp2, timeout=60)
    print(f"Status alternativo: {r.status_code}")
    print(f"Tamanho: {len(r.content):,.0f} bytes")

Status: 404
Tamanho: 26 bytes
Status alternativo: 404
Tamanho: 26 bytes


In [6]:
# Buscar links diretos na página de dados estatísticos da ANP
url_anp_dados = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos"
r = requests.get(url_anp_dados, timeout=30)
soup = BeautifulSoup(r.content, 'html.parser')

# Procurar links de arquivos de vendas
links = []
for a in soup.find_all('a', href=True):
    href = a['href']
    texto = a.text.strip().lower()
    if any(ext in href.lower() for ext in ['.xlsx', '.xls', '.csv', '.zip']):
        if any(termo in texto.lower() or termo in href.lower() 
               for termo in ['venda', 'gasolina', 'combustivel', 'derivado']):
            links.append({'texto': a.text.strip(), 'url': href})

print(f"Links encontrados: {len(links)}")
for l in links[:15]:
    print(f"  {l['texto'][:60]} → {l['url'][-60:]}")

Links encontrados: 244
  Volume de petróleo refinado nas refinarias nacionais (metros → troleo-e-producao-de-derivados/processamento-petroleo-m3.xls
  Volume de petróleo refinado nas refinarias nacionais (barris → etroleo-e-producao-de-derivados/processamento-petroleo-b.xls
  Produção nacional de derivados de petróleo (metros cúbicos) → e-petroleo-e-producao-de-derivados/producao-derivados-m3.xls
  Produção nacional de derivados de petróleo (barris) → de-petroleo-e-producao-de-derivados/producao-derivados-b.xls
  Vendas, pelas distribuidoras, dos derivados combustíveis de  → nteudo/dados-estatisticos/de/vdpb/vendas-combustiveis-m3.xls
  Vendas, pelas distribuidoras, dos derivados combustíveis de  → onteudo/dados-estatisticos/de/vdpb/vendas-combustiveis-b.xls
  2024 → troleo-e-biocombustiveis/asfalto/asfalto-municipio-2024.xlsx
  2023 → troleo-e-biocombustiveis/asfalto/asfalto-municipio-2023.xlsx
  2022 → troleo-e-biocombustiveis/asfalto/asfalto-municipio-2022.xlsx
  2021 → troleo-e-bio

In [5]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas!")

Bibliotecas carregadas!


In [7]:
# URL do arquivo de vendas de combustíveis
url_vendas = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos/de/vdpb/vendas-combustiveis-m3.xls"

r_vendas = requests.get(url_vendas, timeout=60)
print(f"Status: {r_vendas.status_code}")
print(f"Tamanho: {len(r_vendas.content):,.0f} bytes")

# Ler o arquivo
df_anp = pd.read_excel(BytesIO(r_vendas.content))
print(f"\nShape: {df_anp.shape}")
print(f"\nColunas: {df_anp.columns.tolist()}")
print(f"\nPrimeiras linhas:")
df_anp.head(10)

Status: 200
Tamanho: 1,197,259 bytes

Shape: (635, 30)

Colunas: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29']

Primeiras linhas:


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Agência Nacional do Petróleo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,Superintendência de Defesa d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Ver as primeiras linhas sem tratamento
print(df_anp.head(20).to_string())

   Unnamed: 0                                                                     Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  Unnamed: 9  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  Unnamed: 14  Unnamed: 15 Unnamed: 16 Unnamed: 17  Unnamed: 18  Unnamed: 19  Unnamed: 20  Unnamed: 21 Unnamed: 22  Unnamed: 23  Unnamed: 24  Unnamed: 25  Unnamed: 26  Unnamed: 27  Unnamed: 28 Unnamed: 29
0         NaN                                                                            NaN        NaN        NaN        NaN        NaN        NaN        NaN        NaN         NaN          NaN          NaN          NaN          NaN          NaN          NaN         NaN         NaN          NaN          NaN          NaN          NaN         NaN          NaN          NaN          NaN          NaN          NaN          NaN         NaN
1         NaN                                                                            NaN        NaN        NaN        Na

In [11]:
# Verificar abas disponíveis
xl = pd.ExcelFile(BytesIO(r_vendas.content))
print(f"Abas disponíveis: {xl.sheet_names}")

Abas disponíveis: ['Plan1']


In [15]:
for skip in [45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]:
    df_teste = pd.read_excel(BytesIO(r_vendas.content), skiprows=skip)
    primeira_linha = df_teste.iloc[0].tolist()
    valores = [str(v) for v in primeira_linha if str(v) != 'nan']
    print(f"Skip {skip}: {valores[:8]}")

Skip 45: ['BRASIL']
Skip 46: ['COMBUSTÍVEIS TOTAL (m³)']
Skip 47: ['UN. DA FEDERAÇÃO', '(Tudo)']
Skip 48: ['PRODUTO', '(Tudo)']
Skip 49: ['xxxxxxxxxxxxxxxxx', 'xxxxxxxxxxxxxxxxxxxxxxxxx', 'xxxxxxxxxxxxxxxxx', 'xxxxxxxxxxxxxxxxx', 'xxxxxxxxxxxxxxxxx', 'xxxxxxxxxxxxxxxxx', 'xxxxxxxxxxxxxxxxx', 'xxxxxxxxxxxxxxxxx']
Skip 50: ['ANO', 'VARIAÇÃO DO ACUMULADO ']
Skip 51: ['Mês', '2000', '2001', '2002', '2003', '2004', '2005', '2006']
Skip 52: ['Janeiro', '6995110.445727273', '7180755.258491654', '7196582.65419666', '6688221.520586273', '6777775.895675323', '6706311.655030758', '7046949.281047325']
Skip 53: ['Fevereiro', '7416433.1628014855', '6548589.675042676', '6655637.470404449', '6294697.986395178', '6367329.84976438', '6594718.873426799', '6730191.416401953']
Skip 54: ['Março', '7350794.874421148', '7655263.066254173', '7480152.940703152', '6414784.784102043', '7588169.034909088', '7601366.054535048', '7710126.982013308']
Skip 55: ['Abril', '7282572.803630798', '7228914.398111315', '73144

In [16]:
# Ler os dados de vendas totais por mês
df_vendas_raw = pd.read_excel(BytesIO(r_vendas.content), skiprows=51)

# Ver estrutura
print(f"Shape: {df_vendas_raw.shape}")
print(f"\nPrimeiras colunas: {df_vendas_raw.columns.tolist()[:10]}")
print(f"\nPrimeiras linhas:")
print(df_vendas_raw.head(15).to_string())

Shape: (584, 30)

Primeiras colunas: ['Unnamed: 0', 'Unnamed: 1', 'ANO', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']

Primeiras linhas:
   Unnamed: 0    Unnamed: 1              ANO      Unnamed: 3       Unnamed: 4       Unnamed: 5       Unnamed: 6      Unnamed: 7       Unnamed: 8    Unnamed: 9   Unnamed: 10   Unnamed: 11   Unnamed: 12   Unnamed: 13   Unnamed: 14   Unnamed: 15       Unnamed: 16       Unnamed: 17   Unnamed: 18   Unnamed: 19   Unnamed: 20   Unnamed: 21       Unnamed: 22   Unnamed: 23   Unnamed: 24   Unnamed: 25   Unnamed: 26   Unnamed: 27   Unnamed: 28   VARIAÇÃO DO ACUMULADO 
0         NaN           Mês             2000            2001             2002             2003             2004            2005             2006  2.007000e+03  2.008000e+03  2.009000e+03  2.010000e+03  2.011000e+03  2.012000e+03  2.013000e+03              2014              2015  2.016000e+03  2.017000e+03  2.018000e+03  2.019000e+03              

In [20]:
df_vendas = pd.read_excel(BytesIO(r_vendas.content), skiprows=51)

# Converter anos de forma mais robusta
anos_raw = df_vendas.iloc[0, 2:].tolist()
anos = []
for i, v in enumerate(anos_raw):
    try:
        anos.append(str(int(float(str(v)))))
    except:
        anos.append(f'col_{i}')

# Renomear colunas
novas_colunas = ['drop', 'mes'] + anos
df_vendas.columns = novas_colunas
df_vendas = df_vendas.drop('drop', axis=1)

# Remover linha de cabeçalho e pegar só meses
df_vendas = df_vendas.iloc[1:14].copy()
df_vendas = df_vendas[df_vendas['mes'] != 'Total do Ano']

# Só colunas que são anos (dígitos)
cols_anos = [c for c in df_vendas.columns if c.isdigit()]
print(f"Anos disponíveis: {cols_anos}")

# Wide para long
df_vendas_long = df_vendas.melt(
    id_vars=['mes'],
    value_vars=cols_anos,
    var_name='ano',
    value_name='volume_m3'
)

meses_map = {
    'Janeiro': 1, 'Fevereiro': 2, 'Março': 3, 'Abril': 4,
    'Maio': 5, 'Junho': 6, 'Julho': 7, 'Agosto': 8,
    'Setembro': 9, 'Outubro': 10, 'Novembro': 11, 'Dezembro': 12
}

df_vendas_long['mes_num'] = df_vendas_long['mes'].map(meses_map)
df_vendas_long['ano'] = pd.to_numeric(df_vendas_long['ano'], errors='coerce')
df_vendas_long['volume_m3'] = pd.to_numeric(df_vendas_long['volume_m3'], errors='coerce')

df_vendas_long = df_vendas_long[
    (df_vendas_long['ano'] >= 2015) &
    (df_vendas_long['volume_m3'].notna())
].copy()

df_vendas_long['data'] = pd.to_datetime(
    df_vendas_long['ano'].astype(int).astype(str) + '-' +
    df_vendas_long['mes_num'].astype(int).astype(str) + '-01'
)

df_vendas_long = df_vendas_long.sort_values('data').reset_index(drop=True)

print(f"\nShape: {df_vendas_long.shape}")
print(f"Período: {df_vendas_long['data'].min()} até {df_vendas_long['data'].max()}")
print(f"\nAmostra:")
print(df_vendas_long[['data', 'mes', 'volume_m3']].head(12))

Anos disponíveis: ['2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

Shape: (133, 5)
Período: 2015-01-01 00:00:00 até 2026-01-01 00:00:00

Amostra:
         data        mes     volume_m3
0  2015-01-01    Janeiro  1.204793e+07
1  2015-02-01  Fevereiro  1.049025e+07
2  2015-03-01      Março  1.209585e+07
3  2015-04-01      Abril  1.176728e+07
4  2015-05-01       Maio  1.144977e+07
5  2015-06-01      Junho  1.186708e+07
6  2015-07-01      Julho  1.218378e+07
7  2015-08-01     Agosto  1.204688e+07
8  2015-09-01   Setembro  1.198571e+07
9  2015-10-01    Outubro  1.252692e+07
10 2015-11-01   Novembro  1.123985e+07
11 2015-12-01   Dezembro  1.211522e+07


In [21]:
# Salvar dados brutos e tratados
df_vendas_long.to_csv('../output/anp_vendas_combustiveis.csv', index=False)

print("Arquivo salvo!")
print(f"\nResumo ANP:")
print(f"  Período: {df_vendas_long['data'].min().strftime('%b/%Y')} até {df_vendas_long['data'].max().strftime('%b/%Y')}")
print(f"  Total de registros: {len(df_vendas_long)}")

# Comparar volumes 2021 vs 2025
vol_2021 = df_vendas_long[df_vendas_long['ano'] == 2021]['volume_m3'].sum()
vol_2024 = df_vendas_long[df_vendas_long['ano'] == 2024]['volume_m3'].sum()
variacao = ((vol_2024 - vol_2021) / vol_2021) * 100
print(f"\n  Volume total 2021: {vol_2021/1e6:.1f} milhões m³")
print(f"  Volume total 2024: {vol_2024/1e6:.1f} milhões m³")
print(f"  Variação 2021→2024: {variacao:+.1f}%")

Arquivo salvo!

Resumo ANP:
  Período: Jan/2015 até Jan/2026
  Total de registros: 133

  Volume total 2021: 139.4 milhões m³
  Volume total 2024: 156.1 milhões m³
  Variação 2021→2024: +11.9%
